**The Model Using GRU**

In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
import gensim.downloader as api

# Load Dataset
df = pd.read_csv('/content/sample_data/data_berlabel.csv', usecols=['polarity', 'text_akhir']).dropna()

# Balance Dataset
df_majority = df[df['polarity'] == 'negative']
df_minority = [df[df['polarity'] == label] for label in ['neutral', 'positive']]
df_balanced = pd.concat([df_majority] + [resample(df_, replace=True, n_samples=len(df_majority), random_state=42) for df_ in df_minority])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Encode Labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_balanced['polarity']).astype(np.int32)

# Text Preprocessing
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()  # Remove special characters & lowercase
    return text

df_balanced['text_akhir'] = df_balanced['text_akhir'].astype(str).apply(clean_text)

# Tokenization & Padding
vocab_size = 13386
max_length = 100
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(df_balanced['text_akhir'])
X = pad_sequences(tokenizer.texts_to_sequences(df_balanced['text_akhir']), maxlen=max_length, padding='post', truncating='post', dtype=np.int32)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Load GloVe Embeddings
embedding_dim = 50
word_vectors = api.load("glove-twitter-50")
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < vocab_size and word in word_vectors:
        embedding_matrix[i] = word_vectors[word]

# Define Model
from tensorflow.keras.layers import GRU, Bidirectional

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_length, trainable=True),
    #embedding_layer,  # Use the pre-trained GloVe embeddings

    Bidirectional(GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
    Bidirectional(GRU(64, dropout=0.3, recurrent_dropout=0.3)),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])


# Compile & Train Model
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])
model.fit(X_train, y_train, validation_split=0.2, epochs=20, batch_size=32)

# Evaluate Model
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

# Save Model
model.save("gru_sentiment_model.h5")

def predict_sentiment(text, model, tokenizer, max_length=100):
    sequence = pad_sequences(tokenizer.texts_to_sequences([clean_text(text)]), maxlen=max_length, padding='post', truncating='post')
    sentiment_label = np.argmax(model.predict(sequence))
    return label_encoder.inverse_transform([sentiment_label])[0]

# Load & Predict
loaded_model = tf.keras.models.load_model("gru_sentiment_model.h5")
print("Predicted Sentiment:", predict_sentiment("Aplikasi parah, lemot banget!", loaded_model, tokenizer))


[==================================================] 100.0% 199.5/199.5MB downloaded


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 813s 866ms/step - accuracy: 0.4085 - loss: 1.0785 - val_accuracy: 0.5513 - val_loss: 0.9418
Epoch 2/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 817s 883ms/step - accuracy: 0.5247 - loss: 0.9749 - val_accuracy: 0.5899 - val_loss: 0.8712
Epoch 3/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 861s 882ms/step - accuracy: 0.5782 - loss: 0.9037 - val_accuracy: 0.6351 - val_loss: 0.7990
Epoch 4/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 863s 883ms/step - accuracy: 0.6066 - loss: 0.8520 - val_accuracy: 0.6737 - val_loss: 0.7313
Epoch 5/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 802s 872ms/step - accuracy: 0.6427 - loss: 0.7911 - val_accuracy: 0.7015 - val_loss: 0.6819
Epoch 6/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 812s 883ms/step - accuracy: 0.6679 - loss: 0.7415 - val_accuracy: 0.7242 - val_loss: 0.6404
Epoch 7/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 850s 871ms/step - accuracy: 0.6886 - loss: 0.7000 - val_accuracy: 0.7399 - val_loss: 0.6072
Epoch 8/20
919/919 ━━━━━━━━━━━━━━━━━━━━ 857s 930ms/step - accuracy: 0.7079 -

Test Accuracy: 0.8643735647201538


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Predicted Sentiment: negative
